# Causal Analysis Report

Comprehensive causal discovery and validation for the AZIMUTH pipeline.

This notebook replicates the 5 analysis categories from the *causal-chamber-paper*
(juangamella/causal-chamber-paper), adapted for the AZIMUTH process chain:

1. **Ground Truth & Attention-based Discovery** - Extract DAG from SCM and from CausaliT attention weights
2. **Causal Discovery Validation** - Compare estimated vs ground truth with SHD, precision, recall, F1
3. **Interventional Analysis** - Verify do()-intervention effects on downstream variables and F
4. **Out-of-Distribution Analysis** - Test robustness under distributional shift
5. **Symbolic Regression** - Recover structural equations from data

---

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch

# Ensure project root is on path
sys.path.insert(0, '../../')

# Import SCM datasets
from scm_ds.datasets import (
    ds_scm_laser,
    ds_scm_plasma,
    ds_scm_galvanic,
    ds_scm_microetch,
)

# Import causal discovery module
from causaliT.causal_discovery.ground_truth import (
    build_ground_truth_dag,
    get_observable_variables,
    dag_to_edge_list,
)
from causaliT.causal_discovery.metrics import (
    compute_all_metrics,
    compare_graphs,
)
from causaliT.causal_discovery.interventional_analysis import (
    run_intervention_validation,
    run_distributional_comparison,
)
from causaliT.causal_discovery.ood_analysis import run_ood_analysis
from causaliT.causal_discovery.symbolic_analysis import run_symbolic_analysis
from causaliT.causal_discovery.report_generator import generate_causal_report

print('Imports successful.')

In [ ]:
# Process datasets
datasets = {
    'laser': ds_scm_laser,
    'plasma': ds_scm_plasma,
    'galvanic': ds_scm_galvanic,
    'microetch': ds_scm_microetch,
}

SEED = 42
N_SAMPLES = 2000

## 1. Ground Truth DAG

Build the inter-process ground truth adjacency matrix from the SCM definitions.
Only observable variables are considered (input_labels + target_labels + F).

In [ ]:
# Build ground truth DAG
adj_true = build_ground_truth_dag(include_F=True, as_dataframe=True)
variables = get_observable_variables(include_F=True)

print(f'Observable variables ({len(variables)}):')
print(variables)
print(f'\nGround truth edges:')
for parent, child in dag_to_edge_list(adj_true):
    print(f'  {parent} -> {child}')

adj_true

In [ ]:
# Visualise the ground truth DAG as a heatmap
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(adj_true.values, cmap='Blues', aspect='auto')
    ax.set_xticks(range(len(variables)))
    ax.set_yticks(range(len(variables)))
    ax.set_xticklabels(variables, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(variables, fontsize=8)
    ax.set_title('Ground Truth DAG (Adjacency Matrix)', fontsize=12)
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not available for visualization')

## 2. Attention-based Discovery

Load a trained CausaliT/ProT checkpoint, run a forward pass on trajectory data,
and extract attention weights as a causal graph estimate.

> **Note:** This section requires a trained model checkpoint.
> Set `CHECKPOINT_PATH` and `DATA_DIR` below.

In [ ]:
# -- Configure paths (update these for your setup) --
CHECKPOINT_PATH = None  # e.g., 'path/to/checkpoint.pt'
DATA_DIR = None         # e.g., 'path/to/dataset_dir/'

att_maps = None
adj_estimated = None
attention_results = None

if CHECKPOINT_PATH is not None and DATA_DIR is not None:
    import json
    from pathlib import Path
    from causaliT.causal_discovery.attention_discovery import (
        discover_full_dag,
        load_vars_map,
    )
    from causaliT.causal_discovery.discovery_validation import (
        validate_attention_discovery,
    )

    # Load variable maps
    data_dir = Path(DATA_DIR)
    input_vars_map = load_vars_map(data_dir / 'input_vars_map.json')
    target_vars_map = load_vars_map(data_dir / 'target_vars_map.json')

    # Load data
    ds = np.load(data_dir / 'ds.npz')
    input_tensor = torch.tensor(ds['x'][:256], dtype=torch.float32)
    target_tensor = torch.tensor(ds['y'][:256], dtype=torch.float32)

    # Load model (user must adapt this to their config)
    # model = load_model_from_checkpoint(CHECKPOINT_PATH, config)
    # model.eval()

    # -- Uncomment when model is loaded --
    # att_maps, adj_estimated = discover_full_dag(
    #     model=model,
    #     input_tensor=input_tensor,
    #     target_tensor=target_tensor,
    #     input_vars_map=input_vars_map,
    #     target_vars_map=target_vars_map,
    #     threshold=0.1,
    # )
    #
    # attention_results = validate_attention_discovery(
    #     model=model,
    #     input_tensor=input_tensor,
    #     target_tensor=target_tensor,
    #     input_vars_map=input_vars_map,
    #     target_vars_map=target_vars_map,
    #     adj_true=adj_true,
    # )
    # print(attention_results)

else:
    print('No checkpoint configured. Skipping attention-based discovery.')
    print('Set CHECKPOINT_PATH and DATA_DIR above to enable.')

## 3. Interventional Analysis

Use `SCM.do()` to generate data under interventions and verify downstream effects.
Compare reliability F before and after intervention.

In [ ]:
# Run intervention suite
f_comparison = run_intervention_validation(
    datasets=datasets,
    n_samples=N_SAMPLES,
    seed=SEED,
)

print('Intervention effects on F:')
f_comparison

In [ ]:
# Distributional comparisons
dist_comparisons = run_distributional_comparison(
    datasets=datasets,
    n_samples=N_SAMPLES,
    seed=SEED,
)

for label, df in dist_comparisons.items():
    print(f'\n--- {label} ---')
    display(df) if hasattr(__builtins__, 'display') else print(df)

## 4. Out-of-Distribution Analysis

Test pipeline robustness under distributional shifts in process inputs.

In [ ]:
# Run OOD analysis
ood_summary = run_ood_analysis(
    datasets=datasets,
    n_samples=N_SAMPLES,
    seed=SEED,
)

print('OOD Analysis Summary:')
ood_summary

## 5. Symbolic Regression

For each process, test whether the structural equations can be recovered
from (input, output) data using symbolic regression.

In [ ]:
# Run symbolic analysis (polynomial fit fallback)
symbolic_results = run_symbolic_analysis(
    datasets=datasets,
    n_samples=5000,
    seed=SEED,
    use_pysr=False,
    poly_degree=3,
)

print('Symbolic Regression Results:')
symbolic_results

## 6. Generate PDF Report

Compile all analyses into a single PDF report.

In [ ]:
report_path = generate_causal_report(
    output_path='causal_analysis_report.pdf',
    adj_true=adj_true,
    att_maps=att_maps,
    adj_estimated=adj_estimated,
    attention_results=attention_results,
    f_comparison=f_comparison,
    distributional_comparisons=dist_comparisons,
    ood_summary=ood_summary,
    symbolic_results=symbolic_results,
)

print(f'Report generated: {report_path}')